## Refresh EPA American Indian Reservations Data

In [ ]:
import os,sys,psycopg2;
from contextlib import closing;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

lddk = os.path.join(os.path.expanduser('~'),'loading_dock');
src  = 'https://services.arcgis.com/cJ9YHowT8TU7DUyn/ArcGIS/rest/services/American_Indian_Reservations__DOC___EPA_2023_/FeatureServer/1/query';

dbse = os.environ['POSTGRESQL_DB'];
host = os.environ['POSTGRESQL_HOST'];
port = os.environ['POSTGRESQL_PORT'];
user = 'cipsrv';
pswd = os.environ['POSTGRESQL_CIP_PASS'];
thds = 2;


In [ ]:
cs = "dbname=%s user=%s password=%s host=%s port=%s" % (
     dbse
    ,user
    ,pswd
    ,host
    ,port
);

try:
    conn = psycopg2.connect(cs);
except:
    raise Exception("database connection error");

print("database is ready");

In [ ]:
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_support.epa_segs_air CASCADE");
    conn.commit();

print("any preexisting data purged.");

In [ ]:
cmd = '-overwrite -f PostgreSQL '                                                      \
    + 'postgresql://' + user + ':' + pswd + '@' + host + ':' + port + '/' + dbse + ' ' \
    + '"' + src + '?where=1=1&f=json" '                                                \
    + '-nln cipdev_support.epa_segs_air ';

cmd = cmd                                                                              \
    + '-t_srs EPSG:4269 '                                                              \
    + '-lco GEOMETRY_NAME=shape '                                                      \
    + '-lco GEOM_TYPE=geometry '                                                       \
    + '-lco LAUNDER=YES '                                                              \
    + '-lco DIM=2 '                                                                    \
    + '-lco SPATIAL_INDEX=GIST '                                                       \
    + '-nlt GEOMETRY';

z = common.ogr2ogr(
     cmdstring = cmd
);
print(str(z[1]));